# Chương 2: Phân Tích Tăng Trưởng Khu Vực & Phát Hiện Bất Thường Doanh Số

Notebook này thực hiện các yêu cầu phân tích của **Chương 2** trong tài liệu Storytelling:
1. **EDA**: Phân tích xu hướng doanh thu và vẽ ma trận Tăng trưởng vs Quy mô (Growth vs Scale).
2. **Data Mining - Phân cụm (Clustering)**: Sử dụng K-Means để phân cụm lãnh thổ kinh doanh.
3. **Data Mining - Phát hiện bất thường (Anomaly Detection)**: Phân rã chuỗi thời gian và dùng Z-score trên phần dư để tìm các tháng biến động doanh số bất thường.
4. **Lưu kết quả**: Ghi kết quả phân tích ngược lại PostgreSQL DWH.

### 1. Cấu hình & Kết nối Database

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from statsmodels.tsa.seasonal import seasonal_decompose

# Thêm thư mục gốc vào path để import src
sys.path.append(os.path.abspath(os.path.join('..')))
from src.common.database import get_dwh_engine

# Cấu hình biểu đồ
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Kết nối DWH
engine = get_dwh_engine()
print("Kết nối database DWH thành công!")

### 2. Tải Dữ Liệu Phân Tích (EDA)

In [ ]:
# Đọc dữ liệu từ mart sales kpi monthly
query = """
    SELECT 
        m.month_key,
        m.territory_id,
        t.name as territory_name,
        t.countryregioncode as country_code,
        SUM(m.revenue) as revenue,
        SUM(m.orders) as orders,
        SUM(m.quantity) as quantity
    FROM mart.mart_sales_kpi_monthly m
    JOIN staging.salesterritory t ON m.territory_id = t.territoryid
    GROUP BY m.month_key, m.territory_id, t.name, t.countryregioncode
    ORDER BY m.month_key, m.territory_id
"""
df = pd.read_sql_query(query, engine)
print(f"Đã tải {len(df)} dòng dữ liệu.")
df.head()

### 3. Khám Phá Dữ Liệu (EDA)

In [ ]:
# 1. Vẽ xu hướng doanh thu theo khu vực lãnh thổ
df_pivot = df.pivot(index='month_key', columns='territory_name', values='revenue').fillna(0)
df_pivot.plot(kind='line', marker='o', alpha=0.8)
plt.title('Xu hướng doanh thu theo tháng của từng vùng lãnh thổ')
plt.xlabel('Tháng (MonthKey)')
plt.ylabel('Doanh thu (USD)')
plt.legend(title='Vùng lãnh thổ', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Tính tỷ lệ tăng trưởng doanh thu (MoM Growth) và Volatility cho từng khu vực
df['prev_revenue'] = df.groupby('territory_id')['revenue'].shift(1)
df['mom_growth'] = (df['revenue'] - df['prev_revenue']) / df['prev_revenue']
df['mom_growth'] = df['mom_growth'].fillna(0)

# 3. Vẽ ma trận Tăng trưởng vs Quy mô (Growth vs Scale Bubble Chart)
summary = df.groupby('territory_name').agg(
    avg_revenue=('revenue', 'mean'),
    avg_growth=('mom_growth', 'mean'),
    total_orders=('orders', 'sum')
).reset_index()

sns.scatterplot(
    data=summary, 
    x='avg_growth', 
    y='avg_revenue', 
    size='total_orders', 
    hue='territory_name', 
    sizes=(100, 1000), 
    alpha=0.7
)
plt.title('Ma Trận Tăng Trưởng (MoM Growth Rate) vs Quy Mô (Doanh Thu Trung Bình)')
plt.xlabel('Tỷ lệ tăng trưởng trung bình MoM')
plt.ylabel('Doanh thu trung bình mỗi tháng (USD)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 4. Phân Cụm Lãnh Thổ Bằng K-Means

In [ ]:
# Chuẩn bị các đặc trưng phân cụm cho từng vùng
features_df = df.groupby(['territory_id', 'territory_name']).agg(
    mean_revenue=('revenue', 'mean'),
    mean_growth=('mom_growth', 'mean'),
    revenue_volatility=('revenue', 'std'),
    total_orders=('orders', 'sum')
).reset_index()

features_df['revenue_volatility'] = features_df['revenue_volatility'].fillna(0)

# Chuẩn hóa đặc trưng
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_df[['mean_revenue', 'mean_growth', 'revenue_volatility', 'total_orders']])

# Huấn luyện K-Means (k=4 cụm)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
features_df['cluster_label'] = kmeans.fit_predict(scaled_features)

# Đặt tên định hướng kinh doanh cho các cụm
cluster_names = {
    0: "Thị Trường Ổn Định",
    1: "Ngôi Sao Tăng Trưởng",
    2: "Thị Trường Thu Hẹp",
    3: "Thị Trường Tiềm Năng"
}
features_df['cluster_name'] = features_df['cluster_label'].map(cluster_names)
features_df

In [ ]:
# Trực quan hóa kết quả phân cụm
sns.scatterplot(
    data=features_df, 
    x='mean_growth', 
    y='mean_revenue', 
    hue='cluster_name', 
    style='cluster_name',
    s=200, 
    palette='Set1'
)
plt.title('Kết Quả Phân Cụm Lãnh Thổ Kinh Doanh K-Means')
plt.xlabel('Tốc độ tăng trưởng trung bình (MoM)')
plt.ylabel('Doanh thu trung bình tháng (USD)')
plt.legend(title='Phân khúc thị trường')
plt.show()

### 5. Phát Hiện Biến Động Doanh Số Bất Thường (Anomaly Detection)

In [ ]:
# Tạo bảng ghi nhận bất thường
anomaly_results = []

# Phân tích bất thường cho từng khu vực riêng biệt
for tid in df['territory_id'].unique():
    sub_df = df[df['territory_id'] == tid].copy()
    sub_df['date'] = pd.to_datetime(sub_df['month_key'], format='%Y%m')
    sub_df = sub_df.set_index('date').sort_index()
    
    if len(sub_df) < 12:
        # Nếu chuỗi thời gian quá ngắn, tính Z-score trực tiếp trên doanh thu
        rev_mean = sub_df['revenue'].mean()
        rev_std = sub_df['revenue'].std() or 1.0
        sub_df['residual_z'] = (sub_df['revenue'] - rev_mean) / rev_std
    else:
        # Phân rã chuỗi thời gian mùa vụ
        dec = seasonal_decompose(sub_df['revenue'], model='additive', period=12, extrapolate_trend='freq')
        residual = dec.resid
        # Tính Z-score trên phần dư
        sub_df['residual_z'] = (residual - residual.mean()) / (residual.std() or 1.0)
        
    sub_df['is_anomaly'] = np.abs(sub_df['residual_z']) > 2.0
    sub_df = sub_df.reset_index()
    anomaly_results.append(sub_df[['month_key', 'territory_id', 'residual_z', 'is_anomaly']])

anomaly_df = pd.concat(anomaly_results)
df = df.merge(anomaly_df, on=['month_key', 'territory_id'], how='left')
print(f"Tìm thấy {df['is_anomaly'].sum()} tháng biến động bất thường trên toàn bộ khu vực.")
df[df['is_anomaly'] == True].head()

In [ ]:
# Trực quan hóa điểm bất thường của khu vực có nhiều đơn hàng nhất làm ví dụ
example_tid = features_df.sort_values(by='total_orders', ascending=False)['territory_id'].iloc[0]
example_name = features_df[features_df['territory_id'] == example_tid]['territory_name'].iloc[0]

sub_ex = df[df['territory_id'] == example_tid].copy()
plt.plot(sub_ex['month_key'], sub_ex['revenue'], marker='o', label='Doanh thu thực tế')

# Đánh dấu các điểm bất thường màu đỏ
anomalies = sub_ex[sub_ex['is_anomaly'] == True]
plt.scatter(anomalies['month_key'], anomalies['revenue'], color='red', s=150, zorder=5, label='Điểm bất thường')

plt.title(f'Theo dõi điểm biến động doanh số bất thường tại khu vực: {example_name}')
plt.xlabel('Tháng (MonthKey)')
plt.ylabel('Doanh thu (USD)')
plt.legend()
plt.show()

### 6. Ghi Kết Quả Vào Database DWH phục vụ Superset

In [ ]:
# Merge thông tin phân cụm và bất thường
output_db = df.merge(
    features_df[['territory_id', 'cluster_label', 'cluster_name']], 
    on='territory_id', 
    how='left'
)

# Chọn các cột đầu ra
output_db = output_db[[
    'month_key', 'territory_id', 'revenue', 'mom_growth', 
    'cluster_label', 'cluster_name', 'residual_z', 'is_anomaly'
]]

output_db['is_anomaly'] = output_db['is_anomaly'].astype(int)

# Ghi vào schema ml của DWH
output_db.to_sql(
    name='territory_analysis_results', 
    con=engine, 
    schema='ml', 
    if_exists='replace', 
    index=False
)

print("Thành công: Đã ghi nhận kết quả phân tích Chương 2 vào bảng ml.territory_analysis_results!")